Import dataset and libraries


In [135]:
import pandas as pd
import tqdm
import numpy as np
import torch
from torch import nn
import os

In [136]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [137]:
!find /content/drive/MyDrive -iname "masked_sets*"

/content/drive/MyDrive/masked_sets


In [138]:
from pathlib import Path
data_root = Path("/content/drive/MyDrive/masked_sets")

Prepare data 



In [139]:
from sklearn.model_selection import train_test_split

def load_flat_set(root, class_to_idx):
    # Collect paired RGB and depth paths with labels
    X_rgb, X_depth, y = [], [], []

    rgb_paths = sorted([
        p for p in root.glob("*_rgb_*.png")
        if p.is_file()
    ])

    # For each RGB path, check if the corresponding depth path exists
    for rgb_path in rgb_paths:
        depth_path = Path(str(rgb_path).replace("_rgb_", "_depth_"))

        if depth_path.exists():
            # label is everything before "_random_rgb_"
            label = rgb_path.name.split("_random_rgb_")[0]

            if label in class_to_idx:
                X_rgb.append(str(rgb_path))
                X_depth.append(str(depth_path))
                y.append(class_to_idx[label])
        else:
            print(f"Missing depth for {rgb_path}")

    return X_rgb, X_depth, y

# Path for training set
train_root = data_root / "train"

# Build class list from filenames in the flat train folder
classes = sorted({
    p.name.split("_random_rgb_")[0]
    for p in train_root.glob("*_rgb_*.png")
    if p.is_file()
})
class_to_idx = {c: i for i, c in enumerate(classes)}

# Load training set (paired)
X_rgb_all, X_depth_all, y_all = load_flat_set(train_root, class_to_idx)
print("Loaded training samples:", len(y_all))

# Split training set into train/val
X_rgb_train, X_rgb_val, X_depth_train, X_depth_val, y_train, y_val = train_test_split(
    X_rgb_all, X_depth_all, y_all,
    test_size=0.2,
    random_state=1234,
    stratify=y_all
)
print("Train samples:", len(y_train), "Val samples:", len(y_val))

# Paths for test sets
root_low  = data_root / "test" / "low_occlusion"
root_med  = data_root / "test" / "medium_occlusion"
root_high = data_root / "test" / "high_occlusion"

# Load test sets using the same classes & mapping
X_rgb_test_low,  X_depth_test_low,  y_test_low  = load_flat_set(root_low, class_to_idx)
X_rgb_test_med,  X_depth_test_med,  y_test_med  = load_flat_set(root_med, class_to_idx)
X_rgb_test_high, X_depth_test_high, y_test_high = load_flat_set(root_high, class_to_idx)

# Print test set sizes
print("Low occlusion test samples:", len(y_test_low))
print("Med occlusion test samples:", len(y_test_med))
print("High occlusion test samples:", len(y_test_high))

Loaded training samples: 500
Train samples: 400 Val samples: 100
Low occlusion test samples: 200
Med occlusion test samples: 200
High occlusion test samples: 200


Create Fusion Model


In [140]:
import torch
import torch.nn as nn
import torchvision.models as models

class RGBD_Fusion_Model(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # RGB branch
        self.rgb_backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        rgb_feature_dim = self.rgb_backbone.fc.in_features
        self.rgb_backbone.fc = nn.Identity()

        # Depth branch
        self.depth_backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        depth_feature_dim = self.depth_backbone.fc.in_features
        self.depth_backbone.fc = nn.Identity()

        # if depth is 1-channel, change first conv layer
        old_weight = self.depth_backbone.conv1.weight.data
        self.depth_backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.depth_backbone.conv1.weight.data = old_weight.mean(dim=1, keepdim=True)

        # fusion + classifier
        self.classifier = nn.Sequential(
            nn.Linear(rgb_feature_dim + depth_feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, rgb, depth):
        rgb_feat = self.rgb_backbone(rgb)
        depth_feat = self.depth_backbone(depth)

        fused_feat = torch.cat([rgb_feat, depth_feat], dim=1)
        out = self.classifier(fused_feat)
        return out

Train and test CNN models and fuse features from both 

In [141]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for rgb_imgs, depth_imgs, labels in tqdm.tqdm(dataloader):
        rgb_imgs = rgb_imgs.to(device)
        depth_imgs = depth_imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(rgb_imgs, depth_imgs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Total Loss = {total_loss / len(dataloader):.4f}")

def evaluate(model, dataloader, device):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for rgb_imgs, depth_imgs, labels in dataloader:
            rgb_imgs = rgb_imgs.to(device)
            depth_imgs = depth_imgs.to(device)
            labels = labels.to(device)

            outputs = model(rgb_imgs, depth_imgs)
            predicted = torch.argmax(outputs, dim=1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100.0 * correct / total if total > 0 else 0.0
    return acc

In [142]:
# Instantiate separate models for RGB and depth
fusion_model = RGBD_Fusion_Model(num_classes=len(classes)).to(device)

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import cv2
import numpy as np
import torch

print(device)

# fixed image size for batching
IMG_SIZE = (224, 224)

train_aug = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=10, p=0.5),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
], additional_targets={'depth':'image'})


eval_aug = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
], additional_targets={'depth':'image'})

class RGBDepthDataset(Dataset):
    def __init__(self, rgb_paths, depth_paths, labels, aug=None):
        self.rgb_paths = rgb_paths
        self.depth_paths = depth_paths
        self.labels = labels
        self.aug = aug

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        rgb_path = self.rgb_paths[idx]
        depth_path = self.depth_paths[idx]

        # Load RGB
        rgb_image = np.array(Image.open(rgb_path).convert("RGB"))

        # Load depth
        depth_image = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)

        if depth_image is None:
            raise RuntimeError(f"Depth image failed to load: {depth_path}")

        depth_image = depth_image.astype(np.float32)

        if depth_image.max() > 0:
            depth_image = depth_image / depth_image.max()

        # Apply Albumentations
        if self.aug:
            augmented = self.aug(image=rgb_image, depth=depth_image)
            rgb_image = augmented["image"]
            depth_image = augmented["depth"]
        else:
            rgb_image = ToTensorV2()(image=rgb_image)["image"]
            depth_image = torch.tensor(depth_image, dtype=torch.float32)

        # Ensure depth tensor shape is [1,H,W]
        if isinstance(depth_image, np.ndarray):
            depth_image = torch.tensor(depth_image, dtype=torch.float32)

        if depth_image.ndim == 2:
            depth_image = depth_image.unsqueeze(0)

        label = torch.tensor(self.labels[idx], dtype=torch.long)

        return rgb_image, depth_image, label

# Create Datasets
train_dataset = RGBDepthDataset(X_rgb_train, X_depth_train, y_train, aug=train_aug)
val_dataset = RGBDepthDataset(X_rgb_val, X_depth_val, y_val, aug=eval_aug)
test_dataset_low = RGBDepthDataset(X_rgb_test_low, X_depth_test_low, y_test_low, aug=eval_aug)
test_dataset_med = RGBDepthDataset(X_rgb_test_med, X_depth_test_med, y_test_med, aug=eval_aug)
test_dataset_high = RGBDepthDataset(X_rgb_test_high, X_depth_test_high, y_test_high, aug=eval_aug)

# Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader_low = DataLoader(test_dataset_low, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader_med = DataLoader(test_dataset_med, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader_high = DataLoader(test_dataset_high, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# Training loop
train_losses = []
test_accuracies = []
criterion = nn.CrossEntropyLoss()
optimizer_fusion = torch.optim.AdamW(fusion_model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_fusion, mode='max', factor=0.5, patience=2)
best_val_acc = 0.0
best_model_path = "best_fusion_model.pth"
epochs = 20
for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    train(fusion_model, train_loader, criterion, optimizer_fusion, device)
    train_acc = evaluate(fusion_model, train_loader, device)
    print(f"Train accuracy: {train_acc:.2f}%")
    val_acc = evaluate(fusion_model, val_loader, device)
    print(f"Validation Accuracy: {val_acc:.2f}%")
    scheduler.step(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(fusion_model.state_dict(), best_model_path)
        print("Saved best model")

print(f"Best Validation Accuracy: {best_val_acc:.2f}%")

cuda
Epoch 1/30


  0%|          | 0/13 [00:00<?, ?it/s]

100%|██████████| 13/13 [00:03<00:00,  3.32it/s]

Total Loss = 3.9227


Train accuracy: 14.00%
Validation Accuracy: 8.00%
Saved best model
Epoch 2/30


...

Total Loss = 0.0771


Train accuracy: 100.00%
Validation Accuracy: 82.00%
Epoch 30/30


100%|██████████| 13/13 [00:05<00:00,  2.59it/s]

Total Loss = 0.0702


Train accuracy: 100.00%
Validation Accuracy: 82.00%
Best Validation Accuracy: 84.00%


Test the accuracy of the model in low, medium, and high occlusion

In [144]:
fusion_model.load_state_dict(torch.load("best_fusion_model.pth"))
acc_low = evaluate(fusion_model, test_loader_low, device)
acc_med = evaluate(fusion_model, test_loader_med, device)
acc_high = evaluate(fusion_model, test_loader_high, device)

print(f"Low Occlusion Accuracy: {acc_low:.2f}%")
print(f"Medium Occlusion Accuracy: {acc_med:.2f}%")
print(f"High Occlusion Accuracy: {acc_high:.2f}%")

Low Occlusion Accuracy: 81.00%
Medium Occlusion Accuracy: 83.00%
High Occlusion Accuracy: 80.50%


Test the model using real scenes